# 04 — Ultimate Training (max F1, T4-optimized)

Builds on `02_training.ipynb` (best baseline). Adds:
- **PagedAdamW8bit** (bitsandbytes) — 75% optimizer memory cut
- **Cosine** schedule + 10% warmup
- **Label smoothing 0.05** + class weights `(1-freq)^0.5`
- **Multi-sample dropout** classification head (5 passes)
- **EMA** (decay=0.999) — validate w/ EMA weights
- **SWA** last 2 epochs
- **Per-class threshold tuning** post-train (free +1-3 F1)
- **Gradient checkpointing** — fits bs=48 on T4
- maxlen 384 (truncate from 512 cache)

Expected: +3-4 macro F1 over 02 baseline.

## 0 — Install

In [ ]:
import subprocess, sys, platform

IS_APPLE_SILICON = platform.system() == "Darwin" and platform.machine() == "arm64"
IS_COLAB = "google.colab" in sys.modules or "COLAB_GPU" in __import__("os").environ

print(f"Platform      : {platform.system()} {platform.machine()}")
print(f"Colab         : {IS_COLAB}")

pkgs = ["transformers>=4.40", "peft>=0.10", "torch", "pyyaml", "scikit-learn", "tqdm"]
if not IS_APPLE_SILICON:
    pkgs.append("bitsandbytes")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
print("Done.")

## 1 — Config + paths + device

In [ ]:
import os, json, platform, sys, random
from pathlib import Path
import yaml
import torch
import numpy as np

IS_APPLE_SILICON = platform.system() == "Darwin" and platform.machine() == "arm64"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/CSI_Project")
    IS_COLAB = True
    print("Colab — Drive mounted")
except ImportError:
    BASE_DIR = Path(os.path.dirname(os.path.abspath("__file__")))
    if not (BASE_DIR / "datasets").exists():
        BASE_DIR = Path.cwd()
    IS_COLAB = False
    print(f"Local — BASE_DIR: {BASE_DIR}")

cfg_path = BASE_DIR / "config.yaml"
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

# Overrides for ultimate run
MAX_LENGTH       = 384            # truncate from 512 cache → 1.5x speed
EPOCHS           = 10
LR               = 2e-4           # fresh LoRA, cosine decays to 0
WEIGHT_DECAY     = 0.01
WARMUP_RATIO     = 0.10
LABEL_SMOOTHING  = 0.05
EMA_DECAY        = 0.999
FOCAL_GAMMA      = 2.0
MAX_GRAD_NORM    = 1.0
PATIENCE         = 4
MS_DROPOUT_N     = 5              # multi-sample dropout passes
SWA_LAST_N       = 2              # SWA on last N epochs

TOKEN_CACHE_DIR  = BASE_DIR / cfg["token_cache_dir"]
CHECKPOINT_DIR   = BASE_DIR / cfg["checkpoint_dir"]
LOG_DIR          = BASE_DIR / cfg["log_dir"]
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

CACHE_FILE = TOKEN_CACHE_DIR / f"tokens_maxlen{cfg['max_length']}.pt"
assert CACHE_FILE.exists(), f"Missing cache: {CACHE_FILE}"

if torch.cuda.is_available():
    DEVICE = "cuda"
elif IS_APPLE_SILICON and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

SEED = cfg["seed"]
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)

print(f"Device      : {DEVICE}")
print(f"max_length  : {MAX_LENGTH}")
print(f"Epochs      : {EPOCHS}  | LR : {LR}")

## 2 — Dataset + DataLoaders

In [ ]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

class VulnerabilityDataset(Dataset):
    def __init__(self, cache, split, max_length=384):
        if split == "all":
            idx = list(range(len(cache["split_origins"])))
        else:
            idx = [i for i, s in enumerate(cache["split_origins"]) if s == split]
        self.input_ids       = cache["input_ids"][idx][:, :max_length].contiguous()
        self.attention_mask  = cache["attention_mask"][idx][:, :max_length].contiguous()
        self.cwe_labels      = cache["cwe_labels"][idx]
        self.binary_labels   = cache["binary_labels"][idx]
        self.global_ids      = cache["global_ids"][idx]

    def __len__(self): return len(self.input_ids)

    def __getitem__(self, i):
        return {
            "input_ids": self.input_ids[i],
            "attention_mask": self.attention_mask[i],
            "cwe_label": self.cwe_labels[i],
            "binary_label": self.binary_labels[i],
            "global_id": self.global_ids[i],
        }


def make_balanced_sampler(ds):
    labels = ds.cwe_labels.numpy()
    counts = np.bincount(labels, minlength=8)
    w = 1.0 / (counts + 1e-6)
    sw = w[labels]
    return WeightedRandomSampler(torch.tensor(sw, dtype=torch.float),
                                 num_samples=len(labels), replacement=True)


cache = torch.load(CACHE_FILE, weights_only=True)
print(f"Cache: {cache['num_records']:,} records")

BATCH_SIZE = 48 if DEVICE == "cuda" else cfg["batch_size"]

train_ds = VulnerabilityDataset(cache, "train", MAX_LENGTH)
val_ds   = VulnerabilityDataset(cache, "val",   MAX_LENGTH)

_kw = dict(num_workers=2, pin_memory=(DEVICE == "cuda"),
           persistent_workers=False, prefetch_factor=2)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          sampler=make_balanced_sampler(train_ds), **_kw)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **_kw)

# (1 - freq)^0.5 class weights (softer than inverse-freq)
_counts = np.bincount(train_ds.cwe_labels.numpy(), minlength=8).astype(float)
_freq   = _counts / _counts.sum()
CLASS_WEIGHTS = torch.tensor(np.sqrt(1.0 - _freq + 1e-6), dtype=torch.float)
CLASS_WEIGHTS = CLASS_WEIGHTS * (8 / CLASS_WEIGHTS.sum())  # normalize mean=1

print(f"BatchSize     : {BATCH_SIZE}")
print(f"Train batches : {len(train_loader):,} ({len(train_ds):,})")
print(f"Val batches   : {len(val_loader):,} ({len(val_ds):,})")
print(f"Class weights : {CLASS_WEIGHTS.numpy().round(3)}")

## 3 — Model (multi-sample dropout head + focal+LS loss)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel
from peft import LoraConfig, TaskType, get_peft_model

CWE_8_CLASSES = ["CWE-077","CWE-601","CWE-022","CWE-094","CWE-089","CWE-352","CWE-079","unknown"]
CWE_TO_INDEX  = {c: i for i, c in enumerate(CWE_8_CLASSES)}
INDEX_TO_CWE  = {i: c for c, i in CWE_TO_INDEX.items()}


class MultiSampleDropoutHead(nn.Module):
    """LayerNorm → Linear → GELU → [k dropout passes → Linear] avg."""
    def __init__(self, hidden, num_classes=8, dropout=0.2, k=5):
        super().__init__()
        mid = hidden // 2
        self.norm = nn.LayerNorm(hidden)
        self.fc1  = nn.Linear(hidden, mid)
        self.act  = nn.GELU()
        self.dropouts = nn.ModuleList([nn.Dropout(dropout) for _ in range(k)])
        self.fc2  = nn.Linear(mid, num_classes)

    def forward(self, x):
        x = self.act(self.fc1(self.norm(x)))
        if self.training:
            return torch.stack([self.fc2(d(x)) for d in self.dropouts]).mean(0)
        return self.fc2(x)


def focal_label_smooth_loss(logits, targets, class_weights, gamma=2.0, smoothing=0.05):
    n = logits.size(-1)
    logp  = F.log_softmax(logits, dim=-1)
    with torch.no_grad():
        true_dist = torch.full_like(logp, smoothing / (n - 1))
        true_dist.scatter_(1, targets.unsqueeze(1), 1.0 - smoothing)
    ce_per_class = -true_dist * logp                              # (B, C)
    w  = class_weights.to(logits.device).unsqueeze(0)             # (1, C)
    ce_w = (ce_per_class * w).sum(-1)                             # (B,)
    pt = torch.exp(-F.cross_entropy(logits, targets, reduction="none"))
    return (((1 - pt) ** gamma) * ce_w).mean()


class GraphCodeBERTLoRACWEModel(nn.Module):
    def __init__(self, model_name, num_cwe=8, lora_r=16, lora_alpha=32,
                 lora_dropout=0.1, class_weights=None, focal_gamma=2.0,
                 label_smoothing=0.05, ms_drop_n=5, head_dropout=0.2):
        super().__init__()
        encoder = AutoModel.from_pretrained(model_name)
        encoder.config.use_cache = False
        encoder.gradient_checkpointing_enable()
        lora_cfg = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            r=lora_r, lora_alpha=lora_alpha, lora_dropout=lora_dropout,
            target_modules=["query", "key", "value"], bias="none",
        )
        self.encoder = get_peft_model(encoder, lora_cfg)
        # Required for grad checkpointing through PEFT
        if hasattr(self.encoder, "enable_input_require_grads"):
            self.encoder.enable_input_require_grads()
        hidden = self.encoder.config.hidden_size
        self.cwe_head = MultiSampleDropoutHead(hidden, num_cwe, dropout=head_dropout, k=ms_drop_n)
        self.focal_gamma     = focal_gamma
        self.label_smoothing = label_smoothing
        self.register_buffer(
            "class_weights",
            class_weights if class_weights is not None else torch.ones(num_cwe),
        )

    @staticmethod
    def _mean_pool(h, mask):
        m = mask.unsqueeze(-1).float()
        return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)

    def forward(self, input_ids, attention_mask, cwe_labels=None):
        enc    = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self._mean_pool(enc.last_hidden_state, attention_mask)
        logits = self.cwe_head(pooled)
        out = {"logits": logits}
        if cwe_labels is not None:
            out["loss"] = focal_label_smooth_loss(
                logits, cwe_labels, self.class_weights,
                gamma=self.focal_gamma, smoothing=self.label_smoothing,
            )
        return out


model = GraphCodeBERTLoRACWEModel(
    model_name=cfg["model_name"],
    num_cwe=cfg["num_cwe_classes"],
    lora_r=16, lora_alpha=32, lora_dropout=cfg["lora_dropout"],
    class_weights=CLASS_WEIGHTS,
    focal_gamma=FOCAL_GAMMA,
    label_smoothing=LABEL_SMOOTHING,
    ms_drop_n=MS_DROPOUT_N,
).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 4 — Optimizer (PagedAdamW8bit) + cosine scheduler

In [ ]:
from transformers import get_cosine_schedule_with_warmup

trainable_params = [p for p in model.parameters() if p.requires_grad]

USE_8BIT = False
if DEVICE == "cuda":
    try:
        import bitsandbytes as bnb
        optimizer = bnb.optim.PagedAdamW8bit(
            trainable_params, lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.999)
        )
        USE_8BIT = True
        print("Optimizer: PagedAdamW8bit")
    except Exception as e:
        print(f"bnb unavailable ({e}); falling back to AdamW(fused)")
        optimizer = torch.optim.AdamW(trainable_params, lr=LR,
                                      weight_decay=WEIGHT_DECAY, fused=True)
else:
    optimizer = torch.optim.AdamW(trainable_params, lr=LR, weight_decay=WEIGHT_DECAY)
    print("Optimizer: AdamW (cpu/mps)")

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)
print(f"Steps: {total_steps:,} | Warmup: {warmup_steps}")

## 5 — EMA

In [ ]:
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}
        self.backup = {}

    @torch.no_grad()
    def update(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n].mul_(self.decay).add_(p.detach(), alpha=1 - self.decay)

    def apply_to(self, model):
        self.backup = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}
        for n, p in model.named_parameters():
            if p.requires_grad:
                p.data.copy_(self.shadow[n])

    def restore(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad:
                p.data.copy_(self.backup[n])
        self.backup = {}

ema = EMA(model, decay=EMA_DECAY)
print(f"EMA initialized (decay={EMA_DECAY})")

## 6 — Train loop (AMP + EMA + SWA hook)

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from tqdm.auto import tqdm
from copy import deepcopy
import time

USE_AMP = DEVICE == "cuda"
scaler  = torch.amp.GradScaler("cuda", enabled=USE_AMP) if USE_AMP else None
BEST_CKPT = CHECKPOINT_DIR / "ultimate_best.pt"

BEST_F1     = 0.0
no_improve  = 0
history     = []
swa_state   = None  # avg of last N epoch state_dicts
swa_count   = 0


def collect_logits(model, loader):
    model.eval()
    all_logits, all_labels, total_loss, n = [], [], 0.0, 0
    with torch.no_grad():
        for batch in loader:
            ids   = batch["input_ids"].to(DEVICE, non_blocking=True)
            mask  = batch["attention_mask"].to(DEVICE, non_blocking=True)
            lbls  = batch["cwe_label"].to(DEVICE, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                out = model(ids, mask, cwe_labels=lbls)
            total_loss += out["loss"].item(); n += 1
            all_logits.append(out["logits"].float().cpu())
            all_labels.append(lbls.cpu())
    return (torch.cat(all_logits), torch.cat(all_labels), total_loss / max(n, 1))


def metrics(logits, labels):
    preds = logits.argmax(-1).numpy()
    y     = labels.numpy()
    return (f1_score(y, preds, average="macro", zero_division=0),
            precision_score(y, preds, average="macro", zero_division=0),
            recall_score(y, preds, average="macro", zero_division=0),
            preds, y)


def update_swa(swa_state, model, count):
    sd = {k: v.detach().clone() for k, v in model.state_dict().items()}
    if swa_state is None:
        return sd, 1
    new_count = count + 1
    for k in swa_state:
        if swa_state[k].dtype.is_floating_point:
            swa_state[k].mul_(count / new_count).add_(sd[k], alpha=1.0 / new_count)
    return swa_state, new_count


global_step = 0
for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    t0 = time.time()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=True)
    for step, batch in enumerate(pbar, 1):
        ids  = batch["input_ids"].to(DEVICE, non_blocking=True)
        mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        lbls = batch["cwe_label"].to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            out  = model(ids, mask, cwe_labels=lbls)
            loss = out["loss"]
        if USE_AMP:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, MAX_GRAD_NORM)
            scaler.step(optimizer); scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, MAX_GRAD_NORM)
            optimizer.step()
        scheduler.step()
        ema.update(model)
        epoch_loss += loss.item(); global_step += 1
        if step % 50 == 0:
            pbar.set_postfix({"loss": f"{loss.item():.4f}",
                              "lr": f"{scheduler.get_last_lr()[0]:.2e}"})

    train_loss = epoch_loss / len(train_loader)

    # validate w/ EMA weights
    ema.apply_to(model)
    val_logits, val_labels, val_loss = collect_logits(model, val_loader)
    val_f1, val_p, val_r, preds, y = metrics(val_logits, val_labels)
    ema.restore(model)
    elapsed = time.time() - t0

    print(f"\nEpoch {epoch:02d}  train={train_loss:.4f}  val={val_loss:.4f}  "
          f"F1={val_f1:.4f}  P={val_p:.4f}  R={val_r:.4f}  ({elapsed:.0f}s)")
    print(classification_report(y, preds, target_names=CWE_8_CLASSES, zero_division=0))

    history.append({"epoch": epoch, "train_loss": round(train_loss, 4),
                    "val_loss": round(val_loss, 4), "val_f1": round(val_f1, 4),
                    "val_prec": round(val_p, 4), "val_rec": round(val_r, 4),
                    "elapsed_s": round(elapsed, 1)})

    # SWA: accumulate EMA weights for last N epochs
    if epoch > EPOCHS - SWA_LAST_N:
        ema.apply_to(model)
        swa_state, swa_count = update_swa(swa_state, model, swa_count)
        ema.restore(model)
        print(f"  SWA accumulated ({swa_count}/{SWA_LAST_N})")

    if val_f1 > BEST_F1:
        BEST_F1    = val_f1
        no_improve = 0
        torch.save({
            "epoch": epoch, "val_f1": val_f1,
            "model_state_dict": model.state_dict(),
            "ema_shadow": {k: v.cpu() for k, v in ema.shadow.items()},
            "config": cfg,
            "arch": {"lora_r": 16, "lora_alpha": 32, "head": "ms_dropout",
                     "max_length": MAX_LENGTH},
        }, BEST_CKPT)
        print(f"  ✓ New best F1={val_f1:.4f}")
    else:
        no_improve += 1
        print(f"  No improvement ({no_improve}/{PATIENCE})")
        if no_improve >= PATIENCE:
            print(f"Early stop @ epoch {epoch}"); break

print(f"\nDone. Best (EMA, argmax) val F1 = {BEST_F1:.4f}")

## 7 — SWA evaluation (compare vs EMA)

In [ ]:
swa_f1 = None
if swa_state is not None:
    backup = {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(swa_state)
    swa_logits, swa_labels, swa_loss = collect_logits(model, val_loader)
    swa_f1, swa_p, swa_r, _, _ = metrics(swa_logits, swa_labels)
    print(f"SWA   F1={swa_f1:.4f}  P={swa_p:.4f}  R={swa_r:.4f}")
    print(f"EMA   F1={BEST_F1:.4f}")
    if swa_f1 > BEST_F1:
        print("SWA wins — using SWA weights for threshold tune")
        val_logits, val_labels = swa_logits, swa_labels
        BEST_F1 = swa_f1
        torch.save({
            "epoch": "swa", "val_f1": swa_f1,
            "model_state_dict": swa_state,
            "config": cfg,
            "arch": {"lora_r": 16, "lora_alpha": 32, "head": "ms_dropout",
                     "max_length": MAX_LENGTH, "swa": True},
        }, BEST_CKPT)
    else:
        print("EMA wins — restoring")
        model.load_state_dict(backup)
        ema.apply_to(model)
        val_logits, val_labels, _ = collect_logits(model, val_loader)
        ema.restore(model)
else:
    ema.apply_to(model)
    val_logits, val_labels, _ = collect_logits(model, val_loader)
    ema.restore(model)

## 8 — Per-class threshold tuning

In [ ]:
probs = torch.softmax(val_logits, dim=-1).numpy()
y     = val_labels.numpy()
C     = probs.shape[1]

# Per-class threshold sweep — argmax fallback if no class fires
def predict_with_thresholds(probs, thresholds):
    preds = np.full(len(probs), -1, dtype=np.int64)
    fires = probs >= thresholds[None, :]    # (N, C)
    for i in range(len(probs)):
        idx = np.where(fires[i])[0]
        if len(idx) == 0:
            preds[i] = probs[i].argmax()
        else:
            preds[i] = idx[probs[i, idx].argmax()]
    return preds

best_thresh = np.full(C, 0.5)
for c in range(C):
    best_f1_c = -1
    for t in np.arange(0.05, 0.96, 0.02):
        cand = best_thresh.copy(); cand[c] = t
        preds = predict_with_thresholds(probs, cand)
        f1c = f1_score(y, preds, labels=[c], average="macro", zero_division=0)
        if f1c > best_f1_c:
            best_f1_c = f1c; best_thresh[c] = t

tuned_preds = predict_with_thresholds(probs, best_thresh)
tuned_f1    = f1_score(y, tuned_preds, average="macro", zero_division=0)
tuned_p     = precision_score(y, tuned_preds, average="macro", zero_division=0)
tuned_r     = recall_score(y, tuned_preds, average="macro", zero_division=0)

print("Per-class thresholds:")
for c, t in zip(CWE_8_CLASSES, best_thresh):
    print(f"  {c:<10} {t:.2f}")
print(f"\nArgmax F1 : {BEST_F1:.4f}")
print(f"Tuned  F1 : {tuned_f1:.4f}  P={tuned_p:.4f}  R={tuned_r:.4f}")
print(classification_report(y, tuned_preds, target_names=CWE_8_CLASSES, zero_division=0))

# Save thresholds into best ckpt
ckpt = torch.load(BEST_CKPT, map_location="cpu", weights_only=False)
ckpt["thresholds"]     = best_thresh.tolist()
ckpt["val_f1_argmax"]  = float(BEST_F1)
ckpt["val_f1_tuned"]   = float(tuned_f1)
torch.save(ckpt, BEST_CKPT)
print(f"\nSaved thresholds to {BEST_CKPT}")

## 9 — Final report

In [ ]:
import datetime
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y, tuned_preds, labels=list(range(8)))
print("Confusion matrix (rows=true, cols=pred):")
print("            " + "  ".join(f"{c[:6]:>6}" for c in CWE_8_CLASSES))
for i, row in enumerate(cm):
    print(f"{CWE_8_CLASSES[i]:<10} " + "  ".join(f"{v:>6}" for v in row))

log = {
    "run_date": datetime.datetime.now().isoformat(),
    "device": DEVICE,
    "model_name": cfg["model_name"],
    "max_length": MAX_LENGTH,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "optim": "PagedAdamW8bit" if USE_8BIT else "AdamW",
    "val_f1_argmax": float(BEST_F1),
    "val_f1_tuned": float(tuned_f1),
    "swa_f1": float(swa_f1) if swa_f1 is not None else None,
    "thresholds": best_thresh.tolist(),
    "history": history,
}
log_path = LOG_DIR / f"ultimate_run_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(log_path, "w") as f:
    json.dump(log, f, indent=2)
print(f"\nLog saved: {log_path}")

print(f"\n{'Epoch':>5} {'TrainLoss':>10} {'ValLoss':>10} {'ValF1':>8} {'ValP':>8} {'ValR':>8}")
for r in history:
    print(f"{r['epoch']:>5} {r['train_loss']:>10.4f} {r['val_loss']:>10.4f} "
          f"{r['val_f1']:>8.4f} {r['val_prec']:>8.4f} {r['val_rec']:>8.4f}")

## 10 — Inference (apply thresholds)

In [ ]:
from transformers import AutoTokenizer

ckpt = torch.load(BEST_CKPT, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
thresholds = np.array(ckpt["thresholds"])

tokenizer = AutoTokenizer.from_pretrained(cfg["model_name"])
sample = "def get_user(uid): return db.execute('SELECT * FROM users WHERE id=' + uid)"
enc = tokenizer(sample, max_length=MAX_LENGTH, padding="max_length",
                truncation=True, return_tensors="pt")
enc = {k: v.to(DEVICE) for k, v in enc.items()}

with torch.no_grad():
    out = model(enc["input_ids"], enc["attention_mask"])
    p   = torch.softmax(out["logits"], dim=-1)[0].cpu().numpy()

fires = np.where(p >= thresholds)[0]
pred  = (fires[p[fires].argmax()] if len(fires) else p.argmax())

print(f"Code  : {sample[:60]}...")
print(f"Pred  : {INDEX_TO_CWE[int(pred)]}")
print("Top-3 :")
for i in p.argsort()[::-1][:3]:
    print(f"  {INDEX_TO_CWE[int(i)]:<10} prob={p[i]:.4f}  thresh={thresholds[i]:.2f}")